## Load Raw Data

In [269]:
import sqlite3
import pandas as pd
import yaml
import warnings
warnings.filterwarnings('ignore')

# 1. Load SQLite Database
conn = sqlite3.connect('upgradepvp.db')
df_purchases = pd.read_sql_query("SELECT * FROM purchase", conn)
df_deaths = pd.read_sql_query("SELECT * FROM death", conn)
conn.close()

# Ensure datetime format for chronological sorting
df_purchases['created_at'] = pd.to_datetime(df_purchases['created_at'])
df_deaths['created_at'] = pd.to_datetime(df_deaths['created_at'])

# 2. Load YAML Category File
with open('price.yml', 'r') as file:
    yaml_data = yaml.safe_load(file)

print(f"Loaded {len(df_purchases)} purchases and {len(df_deaths)} deaths.")
print("Raw data loaded successfully!")

Loaded 688 purchases and 104 deaths.
Raw data loaded successfully!


## Preprocess Data

This cell transforms the raw database logs into structured transactions that the Association Rule algorithms can understand. Because players can die and lose their items, we cannot simply group all purchases by a single player. Instead, we track their inventory chronologically based on their "lives".

Here is a breakdown of what the code does:

1. **Dynamic Categorization:** It reads the `price.yml` structure and automatically generates a mapping dictionary. This tells the algorithm that a `LEATHER_HELMET` belongs to `Armor_Leather`, or a `BOW` belongs to `Other`, without hardcoding item names.
2. **Chronological Timeline:** It merges the `purchase` and `death` tables into a single timeline, sorted by the exact timestamp the events occurred.
3. **The "Life-Cycle" Logic:** As the loop iterates through the timeline, it tracks the current state of each player's inventory:
    * **Purchases:** When a player buys an item, it is added to their current inventory. If they buy an `ENCHANTED_BOOK`, the code extracts the specific enchantment (e.g., "Power") and drops the generic "Book" tag.
    * **Win Condition:** If a player purchases the `DIAMOND` item, their current transaction is immediately tagged with `Kimenetel: GYŐZELEM` (Outcome: WIN).
    * **Death & Wipes:** When a death event occurs, the player's current inventory is tagged with `Kimenetel: HALÁL` (Outcome: DEATH) and saved. Then, their inventory is completely wiped empty for their next "life" — **unless** they previously purchased a `KeepInventory` item.
4. **1D vs. Multi-Dimensional:** The script generates two distinct datasets simultaneously:
    * **1D Transactions:** Contains only the base item names (e.g., `['WOOD_SWORD', 'SNOW_BALL']`).
    * **Multi-Dimensional Transactions:** Attaches specific dimensions to the data (e.g., `['Tárgy: WOOD_SWORD', 'Kategória: Swords', 'Varázslat: Knockback']`).
5. **Matrix Encoding:** Finally, it uses `TransactionEncoder` to convert these text lists into a True/False binary matrix (One-Hot Encoding), which is the required input format for the `mlxtend` association algorithms.

In [270]:
import re
from mlxtend.preprocessing import TransactionEncoder

# --- Helper Functions ---
def build_category_map(yaml_data):
    cat_map = {}
    for top_cat, content in yaml_data.items():
        if top_cat in ['Win', 'Enchants', 'Other'] or not isinstance(content, dict): continue
        for sub_cat, value in content.items():
            if isinstance(value, dict):
                for piece in value.keys():
                    mat = sub_cat.upper()
                    cat_map[f"{mat}{piece.upper()}"] = f"{top_cat}_{sub_cat}"
                    if mat == "CHAIN": cat_map[f"CHAINMAIL{piece.upper()}"] = f"{top_cat}_{sub_cat}"
                    if mat == "GOLD": cat_map[f"GOLDEN{piece.upper()}"] = f"{top_cat}_{sub_cat}"
            else:
                if top_cat == 'Swords':
                    mat = sub_cat.upper()
                    cat_map[f"{mat}SWORD"] = top_cat
                    if mat == "WOODEN": cat_map["WOODSWORD"] = top_cat
                    if mat == "GOLD": cat_map["GOLDSWORD"] = top_cat
                else:
                    cat_map[sub_cat.upper()] = top_cat
    return cat_map

def parse_item(raw_item):
    if not isinstance(raw_item, str): return str(raw_item), None
    parts = raw_item.split('$')
    base_item = parts[0]
    detail = re.sub(r'§[0-9a-fk-or]', '', parts[1]) if len(parts) > 1 and parts[1] != 'null' else None
    
    # Replace Enchanted Book with the actual enchant
    if base_item == 'ENCHANTED_BOOK' and detail:
        base_item, detail = detail, None  
    return base_item, detail

category_dict = build_category_map(yaml_data)

# --- Combine Events Chronologically ---
events_p = df_purchases[['game_id', 'player_uuid', 'created_at', 'item']].copy()
events_p['event_type'] = 'purchase'
events_d = df_deaths[['game_id', 'dead_uuid', 'created_at']].copy()
events_d.rename(columns={'dead_uuid': 'player_uuid'}, inplace=True)
events_d['event_type'], events_d['item'] = 'death', None
events = pd.concat([events_p, events_d]).sort_values(by=['game_id', 'created_at'])

df_trans_1d: pd.DataFrame = None # type: ignore
df_trans_md: pd.DataFrame = None # type: ignore

# --- Build Transactions ---
def build_transactions(hide_snowball=False):
    global df_trans_1d, df_trans_md
    dataset_1d, dataset_md = [], []
    player_states = {}

    for _, row in events.iterrows():
        key = (row['game_id'], row['player_uuid'])
        if key not in player_states: 
            player_states[key] = {'inv_1d': set(), 'inv_md': set(), 'keep_inv': False}
            
        state = player_states[key]
        
        if row['event_type'] == 'purchase':
            base_item, detail = parse_item(row['item'])

            if hide_snowball and ('SNOW_BALL' in base_item.upper() or 'SNOWBALL' in base_item.upper()):
                continue

            if 'KEEPINVENTORY' in base_item.upper(): state['keep_inv'] = True
            
            # 1D Logic (Items only)
            state['inv_1d'].add(base_item)
            if len(state['inv_1d']) > 0: dataset_1d.append(list(state['inv_1d']))
                
            # Multi-Dimensional Logic
            state['inv_md'].add(f"Tárgy: {base_item}")
            if detail: state['inv_md'].add(f"Varázslat: {detail}")
            
            cat = category_dict.get(base_item.upper().replace('_', ''), None)
            if cat: state['inv_md'].add(f"Kategória: {cat}")
                
            current_tx_md = list(state['inv_md'])
            if base_item == 'DIAMOND': current_tx_md.append("Kimenetel: GYŐZELEM")
            dataset_md.append(current_tx_md)
            
        elif row['event_type'] == 'death':
            if len(state['inv_md']) > 0:
                dataset_md.append(list(state['inv_md']) + ["Kimenetel: HALÁL"])
            if not state['keep_inv']: 
                state['inv_1d'].clear()
                state['inv_md'].clear()

    # --- Encode Dataframes ---
    te_1d = TransactionEncoder()
    df_trans_1d = pd.DataFrame(te_1d.fit(dataset_1d).transform(dataset_1d), columns=te_1d.columns_) # type: ignore

    te_md = TransactionEncoder()
    df_trans_md = pd.DataFrame(te_md.fit(dataset_md).transform(dataset_md), columns=te_md.columns_) # type: ignore

    print("Data preprocessing complete!")
    print(f"1D Transactions: {len(dataset_1d)} | MD Transactions: {len(dataset_md)}")

build_transactions()

Data preprocessing complete!
1D Transactions: 688 | MD Transactions: 777


## Universal Display Function (simple)

This cell defines a helper function, `display_association_rules_simple`, designed to format and present the generated association rules in a clean, human-readable table. 

Here is a breakdown of what the code does:

1. **Metrics Calculation:** It mathematically calculates **Covariance** and **Correlation** for each rule to help evaluate if the item relationships are genuinely meaningful or just statistical coincidences.
2. **Text Formatting:** It converts the algorithm's raw `frozenset` outputs into clean, comma-separated text strings so they are easier to read.
3. **Hungarian Translation:** It selects the most important data columns and renames their headers to Hungarian terms (e.g., *Támasz*, *Konfidencia*, *Lift*).
4. **Deduplication:** It filters out mathematically redundant rules (combinatorial explosions) by dropping rows that have identical Support, Confidence, and Lift values.
5. **Presentation:** Finally, it displays the top results sorted by **Támasz** (Support) in descending order.

In [271]:
import pandas as pd
import numpy as np

def display_association_rules_simple(rules, top_n=15):
    """
    Calculates advanced metrics, cleans frozensets, and displays 
    association rules with Hungarian column headers.
    """
    if rules is None or rules.empty:
        print("No rules found with the current parameters.")
        return

    # 1. Calculate Advanced Metrics (Covariance and Correlation)
    def calc_metrics(row):
        p_A = row['antecedent support']
        p_B = row['consequent support']
        p_AB = row['support']
        
        # Covariance formula
        cov = p_AB - (p_A * p_B)
        
        # Correlation (Phi coefficient) formula
        denom = np.sqrt(p_A * p_B * (1 - p_A) * (1 - p_B))
        corr = cov / denom if denom != 0 else 0
        
        return pd.Series([cov, corr])

    rules[['covariance', 'correlation']] = rules.apply(calc_metrics, axis=1)

    # 2. Clean and format itemsets (remove frozensets for readability)
    rules['clean_antecedents'] = rules['antecedents'].apply(lambda x: ", ".join(list(x)))
    rules['clean_consequents'] = rules['consequents'].apply(lambda x: ", ".join(list(x)))

    # 3. Select columns and rename to Hungarian headers
    formatted_rules = rules[['clean_antecedents', 'clean_consequents', 'support', 'confidence', 'lift', 'covariance', 'correlation']].copy()
    formatted_rules.columns = ['Feltétel (A)', 'Következmény (B)', 'Támasz', 'Konfidencia', 'Lift', 'Kovariancia', 'Korreláció']

    # 4. Remove redundant mathematical combinations (combinatorial explosion)
    formatted_rules = formatted_rules.drop_duplicates(subset=['Támasz', 'Konfidencia', 'Lift'])

    # 5. Display sorted by Lift
    display(formatted_rules.sort_values(by='Támasz', ascending=False).head(top_n))

## Universal Display Function (advanced)

This cell builds upon the simple display function by adding two smart filtering mechanisms to remove "noise" and ensure only the most meaningful rules are shown:

1. **Hierarchical Rule Filtering (Triviality Check):** It actively detects and removes obvious relationships based on item categories or materials. By checking for overlapping core keywords (like "WOOD", "LEATHER", or "SWORD") between the condition and the consequence, it hides trivial rules such as `Swords -> Wooden Sword` or `Armor_Leather -> Leather Chestplate`.
2. **Symmetrical Rule Deduplication:** It identifies reversed rules that represent the exact same item pairing (e.g., `Bow -> Arrow` and `Arrow -> Bow`). It combines the items into a single set and deletes the redundant duplicate, keeping only the logical direction with the highest confidence and lift.

In [272]:
import pandas as pd
import numpy as np

def display_association_rules(rules, top_n=15):
    """
    Calculates advanced metrics, removes reversed/redundant rules, 
    filters out trivial hierarchical subsets, and formats for display.
    """
    if rules is None or rules.empty:
        print("No rules found with the current parameters.")
        return

    # 1. Calculate Advanced Metrics (Covariance and Correlation)
    def calc_metrics(row):
        p_A = row['antecedent support']
        p_B = row['consequent support']
        p_AB = row['support']
        
        cov = p_AB - (p_A * p_B)
        denom = np.sqrt(p_A * p_B * (1 - p_A) * (1 - p_B))
        corr = cov / denom if denom != 0 else 0
        return pd.Series([cov, corr])

    rules[['covariance', 'correlation']] = rules.apply(calc_metrics, axis=1)

    # 2. Detect Trivial / Hierarchical Rules (e.g., Sword -> Wooden Sword)
    def is_trivial(ants, cons):
        # Common core keywords to check for overlap
        core_materials = ["WOOD", "STONE", "IRON", "GOLD", "DIAMOND", "LEATHER", "CHAIN", "SWORD", "HELMET", "CHESTPLATE", "LEGGINGS", "BOOTS"]
        
        for a in ants:
            # Extract base text (e.g., "Kategória: Swords" -> "SWORDS")
            val_a = a.split(':')[-1].strip().upper().replace('_', '')
            for b in cons:
                val_b = b.split(':')[-1].strip().upper().replace('_', '')
                
                # Check 1: Direct substring check (e.g., "SWORD" is inside "WOODSWORD")
                if val_a in val_b or val_b in val_a:
                    return True
                
                # Check 2: Core material overlap (e.g., "ARMORLEATHER" and "LEATHERCHESTPLATE")
                for mat in core_materials:
                    if mat in val_a and mat in val_b:
                        return True
        return False

    rules['is_trivial'] = rules.apply(lambda r: is_trivial(r['antecedents'], r['consequents']), axis=1)
    
    # Drop the trivial rules
    filtered_rules = rules[~rules['is_trivial']].copy()

    if filtered_rules.empty:
        print("All generated rules were filtered out as trivial or redundant.")
        return

    # 3. Handle Symmetrical/Reversed Rules (A -> B vs B -> A)
    # We combine the items into a single unordered set to detect identical combinations
    filtered_rules['combined_items'] = filtered_rules.apply(lambda r: frozenset(r['antecedents'] | r['consequents']), axis=1)

    # Sort by Confidence (descending) and Lift (descending)
    # This ensures that between Bow->Arrow and Arrow->Bow, we keep the one with the highest confidence
    filtered_rules = filtered_rules.sort_values(by=['confidence', 'lift'], ascending=[False, False])

    # Drop the duplicates based on the combined itemset
    filtered_rules = filtered_rules.drop_duplicates(subset=['combined_items'])

    # 4. Clean and format itemsets for human readability
    filtered_rules['clean_antecedents'] = filtered_rules['antecedents'].apply(lambda x: ", ".join(list(x)))
    filtered_rules['clean_consequents'] = filtered_rules['consequents'].apply(lambda x: ", ".join(list(x)))

    # 5. Select columns and rename to Hungarian headers for presentation
    final_table = filtered_rules[['clean_antecedents', 'clean_consequents', 'support', 'confidence', 'lift', 'covariance', 'correlation']].copy()
    final_table.columns = ['Feltétel (A)', 'Következmény (B)', 'Támasz', 'Konfidencia', 'Lift', 'Kovariancia', 'Korreláció']

    # 6. Display final sorted by Támasz
    display(final_table.sort_values(by='Támasz', ascending=False).head(top_n))

## 1D rules

This is mostly just a showcase that these exists, because in this game, we usually buy multiple items together (eg a full leather armor consists of 4 pieces).

### Apriori 1D Rules

In [273]:
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Mine frequent itemsets
freq_items_apriori = apriori(df_trans_1d, min_support=0.05, use_colnames=True)

# 2. Generate rules
rules_apriori = association_rules(freq_items_apriori, metric="confidence", min_threshold=0.4)

# 3. Display results using the universal function
display_association_rules_simple(rules_apriori, top_n=10)

,Feltétel (A),Következmény (B),Támasz,Konfidencia,Lift,Kovariancia,Korreláció
47,SNOW_BALL,IRON_SWORD,0.254360,0.424757,1.322321,0.062001,0.270907
48,IRON_SWORD,SNOW_BALL,0.254360,0.791855,1.322321,0.062001,0.270907
7,Fire Aspect,IRON_SWORD,0.177326,0.924242,2.877280,0.115696,0.629240
8,IRON_SWORD,Fire Aspect,0.177326,0.552036,2.877280,0.115696,0.629240
136,IRON_SWORD,"SNOW_BALL, Fire Aspect",0.155523,0.484163,3.113122,0.105566,0.623830
133,"SNOW_BALL, IRON_SWORD",Fire Aspect,0.155523,0.611429,3.186840,0.106722,0.622339
135,Fire Aspect,"SNOW_BALL, IRON_SWORD",0.155523,0.810606,3.186840,0.106722,0.622339
134,"Fire Aspect, IRON_SWORD",SNOW_BALL,0.155523,0.877049,1.464587,0.049334,0.263532
11,Fire Aspect,SNOW_BALL,0.155523,0.810606,1.353633,0.040630,0.210522
12,GOLD_SWORD,SNOW_BALL,0.155523,0.545918,0.911631,-0.015076,-0.068146


### Messy data (Snowballs)

Snowballs are a very common item that can be purchased in large quantities, but they don't provide meaningful insights for association rules.

When they were included in the dataset, almost all results contained snowballs in one way or another. Let's remove snowballs from our dataset.

In [274]:
build_transactions(hide_snowball=True)

Data preprocessing complete!
1D Transactions: 305 | MD Transactions: 390


### Apriori 1D without Snowballs

The results without snowballs are actually meaningful now, so we will continue with that.

It can now be easily seen that the (as we could have suspected), bows and arrows were bought the most times together.
Different pieces of iron armor were also frequently bought together. However they accumulate most of our results here, that's why we need multi dimensional association rules. 

In [275]:
from mlxtend.frequent_patterns import apriori, association_rules

freq_items_apriori = apriori(df_trans_1d, min_support=0.05, use_colnames=True)
rules_apriori = association_rules(freq_items_apriori, metric="confidence", min_threshold=0.4)
display_association_rules_simple(rules_apriori, top_n=10)

,Feltétel (A),Következmény (B),Támasz,Konfidencia,Lift,Kovariancia,Korreláció
0,BOW,ARROW,0.203279,0.953846,3.985248,0.152271,0.871463
1,ARROW,BOW,0.203279,0.849315,3.985248,0.152271,0.871463
33,IRON_LEGGINGS,IRON_CHESTPLATE,0.154098,0.921569,5.110517,0.123945,0.863925
32,IRON_CHESTPLATE,IRON_LEGGINGS,0.154098,0.854545,5.110517,0.123945,0.863925
37,IRON_LEGGINGS,IRON_HELMET,0.137705,0.823529,5.460358,0.112486,0.842300
38,IRON_HELMET,IRON_LEGGINGS,0.137705,0.913043,5.460358,0.112486,0.842300
30,IRON_CHESTPLATE,IRON_HELMET,0.134426,0.745455,4.942688,0.107229,0.779350
31,IRON_HELMET,IRON_CHESTPLATE,0.134426,0.891304,4.942688,0.107229,0.779350
35,Knockback,IRON_CHESTPLATE,0.131148,0.615385,3.412587,0.092717,0.588905
36,IRON_CHESTPLATE,Knockback,0.131148,0.727273,3.412587,0.092717,0.588905


#### Az "Egymásnak teremtett" páros: Íj és Nyíl
A 1D táblázat legelső, legerősebb szabálya a `BOW` $\Rightarrow$ `ARROW`.
* **Támasz (0.203):** Az összes vásárlási ciklus (élet) több mint **20%**-ában a játékosok íjat és nyilat is vettek. Ez egy nagyon gyakori esemény.
* **Konfidencia (0.953):** Ha egy játékos megvette az íjat, **95.3%** eséllyel nyilat is vett hozzá.
* **Kovariancia (0.152):** A párosítás nemcsak erős (ezt mutatta a Lift), hanem volumenében is meghatározó a szerveren. A játékosok jelentős része erre a stratégiára épít.
* **Lift (3.98):** Az íj megvásárlása közel **négyszeresére** növelte annak az esélyét, hogy a játékos nyilat is vesz, ahhoz képest, mintha vakon tippelnénk a nyílvásárlásra.
* **Korreláció (0.87):** Ez a +1-hez nagyon közeli érték azt mutatja, hogy a két tárgy szinte elválaszthatatlan egymástól (szimmetrikus, erős pozitív kapcsolat).

#### A "Csomagban vásárlás" jelensége: Vas páncélzat
A `IRON_LEGGINGS` $\Rightarrow$ `IRON_CHESTPLATE` (és fordítva) szabályok szintén nagyon beszédesek.
* Itt a **Lift kiemelkedően magas (5.11)**, és a **Korreláció (0.86)** is megközelíti az íj-nyíl párosét.
* *Értelmezés:* Bár a játékosok egyesével veszik meg a páncéldarabokat, a statisztika egyértelműen kimutatja a szándékot: aki az egyik vas páncéldarabot megveszi, az szinte biztosan a teljes "Iron szettet" építi, így a másik darabot is be fogja szerezni. Ezt a magas Kovariancia (0.12) is megerősíti: sokkal gyakrabban fordulnak elő együtt, mint külön.

### Redundant results

However, they are redundant. For example, Bow -> Arrow and Arrow -> Bow are basically the same.

To filter these out, an advanced display function was created above, let's start using that.

In [276]:
display_association_rules(rules_apriori, top_n=10)

,Feltétel (A),Következmény (B),Támasz,Konfidencia,Lift,Kovariancia,Korreláció
0,BOW,ARROW,0.203279,0.953846,3.985248,0.152271,0.871463
45,Knockback,IRON_SWORD,0.131148,0.615385,2.536383,0.079441,0.452543
36,IRON_CHESTPLATE,Knockback,0.131148,0.727273,3.412587,0.092717,0.588905
242,"IRON_LEGGINGS, IRON_CHESTPLATE",Knockback,0.121311,0.787234,3.693944,0.088471,0.598382
44,IRON_LEGGINGS,Knockback,0.121311,0.725490,3.404223,0.085676,0.560653
19,Fire Aspect,Knockback,0.114754,0.777778,3.649573,0.083311,0.573649
16,Fire Aspect,IRON_SWORD,0.114754,0.777778,3.205706,0.078957,0.519370
259,"IRON_LEGGINGS, IRON_HELMET",Knockback,0.108197,0.785714,3.686813,0.078850,0.558773
650,"IRON_LEGGINGS, IRON_CHESTPLATE, IRON_HELMET",Knockback,0.108197,0.825000,3.871154,0.080247,0.580517
232,"IRON_CHESTPLATE, IRON_HELMET",Knockback,0.108197,0.804878,3.776735,0.079549,0.569476


### FP-Growth 1D Rules

The results should be the same as the Apriori algorithm.

In [277]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

freq_items_1d = fpgrowth(df_trans_1d, min_support=0.05, use_colnames=True)
rules_1d = association_rules(freq_items_1d, metric="confidence", min_threshold=0.4)
display_association_rules(rules_1d, top_n=10)

,Feltétel (A),Következmény (B),Támasz,Konfidencia,Lift,Kovariancia,Korreláció
24,BOW,ARROW,0.203279,0.953846,3.985248,0.152271,0.871463
228,Knockback,IRON_SWORD,0.131148,0.615385,2.536383,0.079441,0.452543
174,IRON_CHESTPLATE,Knockback,0.131148,0.727273,3.412587,0.092717,0.588905
209,"IRON_LEGGINGS, IRON_CHESTPLATE",Knockback,0.121311,0.787234,3.693944,0.088471,0.598382
186,IRON_LEGGINGS,Knockback,0.121311,0.725490,3.404223,0.085676,0.560653
34,Fire Aspect,Knockback,0.114754,0.777778,3.649573,0.083311,0.573649
31,Fire Aspect,IRON_SWORD,0.114754,0.777778,3.205706,0.078957,0.519370
402,"IRON_LEGGINGS, IRON_HELMET",Knockback,0.108197,0.785714,3.686813,0.078850,0.558773
408,"IRON_LEGGINGS, IRON_CHESTPLATE, IRON_HELMET",Knockback,0.108197,0.825000,3.871154,0.080247,0.580517
396,"IRON_CHESTPLATE, IRON_HELMET",Knockback,0.108197,0.804878,3.776735,0.079549,0.569476


## Multi-Dimensional rules

Here **FP-Growth** is run with different parameters. Apriori and Apriori-TID would yield the same results (as shown above), so there's no reason to run those as well.

In [278]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

freq_items_md = fpgrowth(df_trans_md, min_support=0.1, use_colnames=True)
rules_md = association_rules(freq_items_md, metric="confidence", min_threshold=0.4)
display_association_rules_simple(rules_md, top_n=15)

,Feltétel (A),Következmény (B),Támasz,Konfidencia,Lift,Kovariancia,Korreláció
3,Tárgy: GOLD_SWORD,Kategória: Swords,0.320513,1.000000,1.153846,0.042735,0.269386
23,Tárgy: IRON_SWORD,Kategória: Swords,0.217949,1.000000,1.153846,0.029060,0.207063
1,Kimenetel: HALÁL,Kategória: Swords,0.207692,0.952941,1.099548,0.018803,0.133982
0,Tárgy: WOOD_SWORD,Kategória: Swords,0.187179,1.000000,1.153846,0.024957,0.188224
196,Tárgy: Knockback,Kategória: Swords,0.174359,0.944444,1.089744,0.014359,0.108871
5,Tárgy: BOW,Tárgy: ARROW,0.166667,0.955882,4.905186,0.132689,0.882897
4,Tárgy: ARROW,Tárgy: BOW,0.166667,0.855263,4.905186,0.132689,0.882897
14,Kategória: Armor_Leather,Kategória: Swords,0.161538,0.875000,1.009615,0.001538,0.011665
32,Kategória: Armor_Iron,Kategória: Swords,0.161538,0.913043,1.053512,0.008205,0.063252
41,Kategória: Armor_Iron,Tárgy: IRON_CHESTPLATE,0.158974,0.898551,5.652174,0.130848,0.937750


### Filter trivial results

The results contain "trivial" lines, such as Gold Sword -> Category Sword.

To circumvent these trivial results, a hierarchy filter has been added to the advanced display function. Let's use that from now on.

In [279]:
display_association_rules(rules_md, top_n=15)

,Feltétel (A),Következmény (B),Támasz,Konfidencia,Lift,Kovariancia,Korreláció
1,Kimenetel: HALÁL,Kategória: Swords,0.207692,0.952941,1.099548,0.018803,0.133982
196,Tárgy: Knockback,Kategória: Swords,0.174359,0.944444,1.089744,0.014359,0.108871
5,Tárgy: BOW,Tárgy: ARROW,0.166667,0.955882,4.905186,0.132689,0.882897
14,Kategória: Armor_Leather,Kategória: Swords,0.161538,0.875000,1.009615,0.001538,0.011665
32,Kategória: Armor_Iron,Kategória: Swords,0.161538,0.913043,1.053512,0.008205,0.063252
42,Tárgy: IRON_CHESTPLATE,Kategória: Swords,0.148718,0.935484,1.079404,0.010940,0.088016
46,"Tárgy: IRON_CHESTPLATE, Kategória: Armor_Iron",Kategória: Swords,0.148718,0.935484,1.079404,0.010940,0.088016
12,Tárgy: ARROW,Kategória: Swords,0.141026,0.723684,0.835020,-0.027863,-0.206933
6,Tárgy: BOW,Kategória: Swords,0.138462,0.794118,0.916290,-0.012650,-0.098076
9,"Tárgy: BOW, Kategória: Swords",Tárgy: ARROW,0.133333,0.962963,4.941520,0.106351,0.777379


#### A "Konfidencia csapdája": Halál és a Kardok
A többdimenziós (MD) eredmények között található a tökéletes példa arra, amiért a Liftet és a Korrelációt fel kell találni. Nézzük a `Kimenetel: HALÁL` $\Rightarrow$ `Kategória: Swords` szabályt:
* **Konfidencia (0.952):** Aki meghal a játékban, annál **95.2%**-ban van valamilyen kard. Ha csak ezt néznénk, azt hihetnénk, hogy a kard okozza a halált, vagy a kettő szorosan összefügg.
* **Lift (1.09) és Korreláció (0.13):** Ezek az értékek viszont megmutatják a valóságot. A Lift éppen csak 1 felett van, a Korreláció pedig közelít a nullához. 
* *Értelmezés:* A kard birtoklása szinte teljesen **független** attól, hogy a játékos meghal-e. A konfidencia kizárólag azért ilyen magas, mert ebben a játékban gyakorlatilag *mindenkinél* van kard (az alapsokaságban nagyon magas a támogatottsága). A metrikák helyesen azonosították, hogy ez egy statisztikailag érdektelen (triviális) összefüggés.

#### Nullához közeli kovariancia (A "Zaj" és a függetlenség)
* **Szabály:** `Kategória: Armor_Leather` $\Rightarrow$ `Kategória: Swords`
* **Kovariancia:** **0.0015**
* **Mit jelent ez a játékban?** Itt a konfidencia elég magas (**87.5%**), tehát a bőrpáncélt vevők szinte mindig vesznek kardot is. De a kovariancia értéke gyakorlatilag nulla. Tehát a bőrpáncél megvásárlása az semmilyen ráhatással nincs arra, hogy a játékos vesz-e kardot. Csak azért vannak gyakran egy kosárban, mert mindkettő tipikus, olcsó "kezdő" felszerelés, amit amúgy is mindenki megvesz. Nincs közöttük rejtett játékmechanikai kapocs.

### Decrease min_support

Most of the consequences are `Category: Swrods`. That's mostly because everyone buys a sword at the beginning of the game.

Let's decrease min_support and display lot's of results. This is a small dataset, so we can find meaningful results manually.

In [280]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

freq_items_md = fpgrowth(df_trans_md, min_support=0.05, max_len=3, use_colnames=True)
rules_md = association_rules(freq_items_md, metric="confidence", min_threshold=0.4)
display_association_rules(rules_md, top_n=150)

,Feltétel (A),Következmény (B),Támasz,Konfidencia,Lift,Kovariancia,Korreláció
2,Kimenetel: HALÁL,Kategória: Swords,0.207692,0.952941,1.099548,0.018803,0.133982
323,Tárgy: Knockback,Kategória: Swords,0.174359,0.944444,1.089744,0.014359,0.108871
35,Tárgy: BOW,Tárgy: ARROW,0.166667,0.955882,4.905186,0.132689,0.882897
205,Kategória: Armor_Iron,Kategória: Swords,0.161538,0.913043,1.053512,0.008205,0.063252
54,Kategória: Armor_Leather,Kategória: Swords,0.161538,0.875000,1.009615,0.001538,0.011665
227,Tárgy: IRON_CHESTPLATE,Kategória: Swords,0.148718,0.935484,1.079404,0.010940,0.088016
232,"Tárgy: IRON_CHESTPLATE, Kategória: Armor_Iron",Kategória: Swords,0.148718,0.935484,1.079404,0.010940,0.088016
50,Tárgy: ARROW,Kategória: Swords,0.141026,0.723684,0.835020,-0.027863,-0.206933
36,Tárgy: BOW,Kategória: Swords,0.138462,0.794118,0.916290,-0.012650,-0.098076
39,"Tárgy: BOW, Kategória: Swords",Tárgy: ARROW,0.133333,0.962963,4.941520,0.106351,0.777379


## Further improvements

- Categorize enchantments (for example, sharpness can only be applied to swords, check what enchantments people buy for different swords)
- Categorize armor & sword together (eg iron sword - iron armor)
- Analyzing specific groups (eg. What was the most used armor with Iron Sword?)
- Ensure that the consequence is not a category.